# A.X.I.S — Autonomous eXecution & Intelligence System

Run each cell **top to bottom**. After Cell 5 you will get a public URL — open it in any browser.

| Component | Details |
|---|---|
| LLM | Ollama llama3:8b on Colab T4 GPU |
| Memory | ChromaDB persistent vector store |
| Backend | FastAPI + Socket.io |
| Public URL | ngrok HTTPS tunnel |

In [ ]:
# ============================================================
# CELL 1 — Direct Download Ollama (Reliable Docker/Colab Method)
# ============================================================
import os
import requests
import subprocess
import time

OLLAMA_BIN = '/usr/local/bin/ollama'
os.environ['PATH'] = '/usr/local/bin:' + os.environ.get('PATH', '')

if not os.path.exists(OLLAMA_BIN):
    print('Downloading Ollama Linux binary directly...')
    # Colab is a Docker container (no systemd), so downloading the raw binary is most reliable.
    url = 'https://ollama.com/download/ollama-linux-amd64'
    response = requests.get(url, stream=True)
    response.raise_for_status()
    
    with open(OLLAMA_BIN, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
                
    os.chmod(OLLAMA_BIN, 0o755)
    print('✅ Ollama binary successfully installed.')
else:
    print('✅ Ollama binary already exists.')

print('Starting Ollama background server...')
subprocess.Popen(
    [OLLAMA_BIN, 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(6)  # Wait for server to bind to port 11434

try:
    r = requests.get('http://localhost:11434')
    print('✅ Ollama is running:', r.text.strip())
except Exception as e:
    print('❌ Ollama not responding:', e)

In [ ]:
# ============================================================
# CELL 2 — Pull the LLM model
# ============================================================
MODEL = 'llama3:8b'

print(f'Pulling {MODEL}...')
result = subprocess.run([OLLAMA_BIN, 'pull', MODEL], capture_output=True, text=True)
print(result.stdout[-500:] if result.stdout else 'Done')

models = subprocess.run([OLLAMA_BIN, 'list'], capture_output=True, text=True)
print('Installed models:\n', models.stdout)

In [ ]:
# ============================================================
# CELL 3 — Install Python packages
# ============================================================
print('Installing Python dependencies...')
os.system('pip install -q fastapi uvicorn python-socketio chromadb pyngrok requests')
print('Done.')

In [ ]:
# ============================================================
# CELL 4 — Clone the AXIS_AI project from GitHub
# ============================================================
REPO_URL = 'https://github.com/aaweshdas/AXIS-AI.git'

if not os.path.exists('/content/AXIS_AI'):
    print('Cloning A.X.I.S project...')
    os.system(f'git clone {REPO_URL} /content/AXIS_AI')
else:
    print('Project already cloned, pulling latest...')
    os.system('git -C /content/AXIS_AI pull')

os.chdir('/content/AXIS_AI')
print('Working directory:', os.getcwd())
print('Files:', os.listdir('.'))

In [ ]:
# ============================================================
# CELL 5 — Start FastAPI + Socket.io and open ngrok tunnel
# ============================================================
import threading
import sys
import uvicorn
from pyngrok import ngrok

sys.path.insert(0, '/content/AXIS_AI')
from server import socket_app, MODEL

PORT = 8000

def run_server():
    uvicorn.run(socket_app, host='0.0.0.0', port=PORT, log_level='warning')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)

public_url = ngrok.connect(PORT).public_url

print('\n' + '='*55)
print('  A.X.I.S is LIVE:')
print(f'  {public_url}/ui')
print('='*55)
print(f'  Model  : {MODEL}')
print(f'  Health : {public_url}/api/health')
print('='*55 + '\n')

# Patch the frontend to use the public ngrok URL
html_path = '/content/AXIS_AI/static/index.html'
with open(html_path, 'r') as f:
    html = f.read()

patch = f'\n  <script>window.AXIS_SERVER_URL = "{public_url}";</script>\n'
if 'AXIS_SERVER_URL' not in html:
    html = html.replace(
        '<script src="https://cdn.socket.io',
        patch + '  <script src="https://cdn.socket.io'
    )
    with open(html_path, 'w') as f:
        f.write(html)

print('Frontend patched. Open the URL above in your browser!')

In [ ]:
# ============================================================
# CELL 6 — Health check
# ============================================================
import requests as req

print('Health checks:\n')

try:
    r = req.get('http://localhost:11434', timeout=3)
    print('Ollama :', r.text.strip())
except Exception:
    print('Ollama : NOT reachable')

try:
    r = req.get(f'http://localhost:{PORT}/api/health', timeout=5)
    d = r.json()
    print(f'FastAPI: online | model={d["model"]} | memories={d["memory_entries"]}')
except Exception as e:
    print(f'FastAPI: {e}')

try:
    tunnels = ngrok.get_tunnels()
    for t in tunnels:
        print(f'ngrok  : {t.public_url} -> port {PORT}')
except Exception:
    print('ngrok  : no active tunnels')

print('\nOpen the /ui URL from Cell 5 in your browser!')